# Experiment 10: a neural model, and the last diversity hypothesis

**This notebook runs on Kaggle, not locally.** There is no local GPU. Upload it, attach
the competition data, set the accelerator to GPU, and Run All.

## Why this is the only lever left

Everything else has been closed, and closed with measurements rather than opinions:

| direction | result |
|---|---|
| LightGBM hyperparameters | exhausted, +0.008328 then flat |
| feature engineering | rejected, the generator's rules cap at AUC 0.835 |
| missingness indicators | rejected, no target lift outside sampling error |
| external data | 7,500 rows against 691,369, itself synthetic |
| capacity diversity within LightGBM | blends negative, `07` |
| CatBoost, cross-GBDT diversity | blend -0.000250, rejected, `06` |
| stochastic diversity, bagged seeds | works, saturates at 4 seeds, `09` |

Best model is a 5-seed bagged LightGBM blend at **CV 0.963880, LB 0.965080**. The
leaders are at 0.97102.

The case for a neural model is the generator forensics rather than optimism. The
synthetic data is a smooth, roughly additive, calibrated field with no leak, no duplicate
rows, and a monotone response on nearly every feature. That is the territory where
neural nets are competitive with GBDTs instead of losing to them, and it is a genuinely
different inductive bias rather than a third histogram booster.

## What counts as success

**Not the neural CV on its own.** It will very likely land below 0.9639 and that is not
failure. `06` and `07` established the decision rule the expensive way:

> Gate on the measured rank-blend AUC against the best single model. Not on rank
> correlation, which was anti-predictive in both directions on this data. CatBoost
> correlated at 0.9877 and blended negatively; bagged seeds correlate at 0.990 to 0.996
> and blend positively. What matters is whether the members are comparably strong.

So the number to watch is the blend, and the bar is a gain over 0.963880 that wins folds
consistently. A neural model landing anywhere near 0.960 with genuinely different errors
would be a better blend partner than CatBoost at 0.9622 was.

## The one thing that must not go wrong

The out-of-fold vector has to align **row for row** with the thirteen vectors already on
disk locally, or it cannot be blended or compared with any of them. Cell 2 verifies the
fold split reproduces the local one exactly, by checksum, before anything trains.

In [ ]:
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer

SEED = 42
N_SPLITS = 5
TARGET, ID = "addicted_label", "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# Reference numbers from the local repo, for the comparisons at the end.
LGB_BEST_CV = 0.963880      # 5-seed bagged blend, exp 15
LGB_EXP8_CV = 0.963275      # best single model, exp 8

# SMOKE trims everything to a couple of minutes on CPU. Run it once before spending a
# GPU session: it exercises every line of the training loop on a small subset, so a
# crash costs two minutes instead of eight and a wasted session.
SMOKE = False

EPOCHS, BATCH, LR, WD = 30, 4096, 3e-3, 1e-4
PATIENCE = 5
EMB_DIM, HIDDEN, DROPOUT = 8, (512, 256, 128), 0.20
USE_MISSING_MASK = True

if SMOKE:
    EPOCHS, N_SPLITS = 2, 2

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)


seed_all(SEED)
print("torch", torch.__version__, "| device", DEV)
if DEV.type != "cuda":
    print("\nWARNING: no GPU. Set the accelerator in the notebook settings, or this")
    print("will take roughly an hour instead of ten minutes.")

In [ ]:
def locate():
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    raise FileNotFoundError("could not find train.csv")


OUT, RAW = locate()
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")
if SMOKE:
    train = train.sample(20000, random_state=SEED).reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    sample = sample.head(5000).reset_index(drop=True)
    print("SMOKE MODE: 20k train rows, 2 folds, 2 epochs. Numbers are meaningless.")

FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
y = train[TARGET].to_numpy().astype(np.float32)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i

# The alignment gate. These were computed locally on 2026-08-04 with scikit-learn
# 1.9.0. If any of them disagree, the out-of-fold vector this notebook produces cannot
# be blended with the local ones and the run is worthless. Stop and reconcile.
EXPECT = {"rows": 691369, "rate": 0.709424, "sha": "ec282b0968059676",
          "sizes": [138274, 138274, 138274, 138274, 138273],
          "first20": [3, 3, 3, 4, 2, 3, 4, 0, 3, 4, 1, 1, 2, 1, 3, 1, 1, 4, 0, 3]}
got = {"rows": len(train), "rate": round(float(y.mean()), 6),
       "sha": hashlib.sha256(folds.tobytes()).hexdigest()[:16],
       "sizes": np.bincount(folds).tolist(), "first20": folds[:20].tolist()}

ALIGNED = True
for k in ([] if SMOKE else EXPECT):
    ok = got[k] == EXPECT[k]
    ALIGNED &= ok
    print(f"  [{'ok' if ok else 'MISMATCH'}] {k}")
    if not ok:
        print(f"        expected {EXPECT[k]}")
        print(f"        got      {got[k]}")

print(f"\nfold alignment: {'verified' if ALIGNED else 'FAILED, do not use this OOF'}")
print(f"sklearn {__import__('sklearn').__version__}, expected 1.9.0 locally")
print(f"\n{len(train):,} train rows, {len(test):,} test rows, {len(FEATURES)} features")

## Preprocessing, and one judgement call about missingness

LightGBM routes NaN natively. A neural net cannot, so the numeric columns have to be
imputed, and imputation invents a value that was never observed.

`04_missingness_diagnostic.ipynb` established that missingness carries **no signal about
the target**: not one of the twelve features shows a lift outside sampling error, and the
missing-count correlation is +0.0025. That finding is not a reason to drop the mask here,
because the mask is doing a different job. It is not telling the model that a missing
value predicts the target. It is telling the model which inputs are real and which are
median fill, so the network is not misled into treating an invented value as an
observation. Set `USE_MISSING_MASK = False` to test that reasoning rather than trust it.

Quantile transform to a normal output rather than standardisation, because several of
these columns are skewed and neural nets are far more sensitive to that than trees are.
It is fit **inside the fold loop**, on the training rows only. Fitting it on all of train
would leak validation distribution into the model, which is exactly the class of mistake
the leak checklist exists for.

In [ ]:
# Categorical encoding is fit on train and test together, which is safe: it uses no
# target information whatsoever, only the set of levels that exist.
cat_codes_tr, cat_codes_te, cat_sizes = {}, {}, []
for c in CAT_COLS:
    both = pd.concat([train[c], test[c]], ignore_index=True).astype("object")
    levels = sorted(both.dropna().unique().tolist())
    lut = {v: i + 1 for i, v in enumerate(levels)}   # 0 is reserved for missing
    cat_codes_tr[c] = train[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_codes_te[c] = test[c].map(lut).fillna(0).to_numpy().astype(np.int64)
    cat_sizes.append(len(levels) + 1)
    print(f"{c}: {len(levels)} levels plus a missing slot")

Xc_tr = np.stack([cat_codes_tr[c] for c in CAT_COLS], axis=1)
Xc_te = np.stack([cat_codes_te[c] for c in CAT_COLS], axis=1)

num_tr_raw = train[NUM_COLS].to_numpy().astype(np.float32)
num_te_raw = test[NUM_COLS].to_numpy().astype(np.float32)
mask_tr = np.isnan(num_tr_raw).astype(np.float32)
mask_te = np.isnan(num_te_raw).astype(np.float32)
print(f"\n{len(NUM_COLS)} numeric columns, mean missing rate "
      f"{mask_tr.mean():.4f} train / {mask_te.mean():.4f} test")

In [ ]:
class TabMLP(nn.Module):
    def __init__(self, n_num, cat_sizes, emb_dim=EMB_DIM, hidden=HIDDEN, p=DROPOUT):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(s, emb_dim) for s in cat_sizes])
        dim = n_num + emb_dim * len(cat_sizes)
        layers = []
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(p)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, xn, xc):
        e = [emb(xc[:, i]) for i, emb in enumerate(self.embs)]
        return self.net(torch.cat([xn] + e, dim=1)).squeeze(1)


def make_loader(xn, xc, yy, bs, shuffle, drop_last=False):
    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(xn), torch.from_numpy(xc),
        torch.from_numpy(yy) if yy is not None else torch.zeros(len(xn)))
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle,
                                       num_workers=2, pin_memory=True,
                                       drop_last=drop_last)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xn, xc, _ in loader:
        out.append(torch.sigmoid(model(xn.to(DEV), xc.to(DEV))).float().cpu().numpy())
    return np.concatenate(out)


print("model defined")

In [ ]:
oof = np.zeros(len(train), dtype=np.float64)
test_pred = np.zeros(len(test), dtype=np.float64)
fold_scores = []
t_start = time.time()

for f in range(N_SPLITS):
    seed_all(SEED + f)
    tr_i = np.where(folds != f)[0]
    va_i = np.where(folds == f)[0]

    # Fit imputation and the quantile transform on training rows only.
    med = np.nanmedian(num_tr_raw[tr_i], axis=0)
    def prep(raw, mask):
        x = np.where(np.isnan(raw), med, raw)
        return x
    qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                             subsample=200_000, random_state=SEED)
    qt.fit(prep(num_tr_raw[tr_i], mask_tr[tr_i]))

    def build(raw, mask):
        x = qt.transform(prep(raw, mask)).astype(np.float32)
        return np.hstack([x, mask]) if USE_MISSING_MASK else x

    Xn_tr, Xn_va = build(num_tr_raw[tr_i], mask_tr[tr_i]), build(num_tr_raw[va_i], mask_tr[va_i])
    Xn_te = build(num_te_raw, mask_te)

    tr_loader = make_loader(Xn_tr, Xc_tr[tr_i], y[tr_i], BATCH, True, drop_last=True)
    va_loader = make_loader(Xn_va, Xc_tr[va_i], None, BATCH * 4, False)
    te_loader = make_loader(Xn_te, Xc_te, None, BATCH * 4, False)

    model = TabMLP(Xn_tr.shape[1], cat_sizes).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(tr_loader))
    lossf = nn.BCEWithLogitsLoss()

    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        for xn, xc, yy in tr_loader:
            opt.zero_grad(set_to_none=True)
            loss = lossf(model(xn.to(DEV), xc.to(DEV)), yy.to(DEV))
            loss.backward()
            opt.step()
            sched.step()
        auc = roc_auc_score(y[va_i], predict(model, va_loader))
        if auc > best_auc:
            best_auc, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep % 5 == 0 or bad >= PATIENCE:
            print(f"  fold {f} epoch {ep:>2}: val AUC {auc:.6f} (best {best_auc:.6f})")
        if bad >= PATIENCE:
            print(f"  early stop at epoch {ep}")
            break

    model.load_state_dict(best_state)
    oof[va_i] = predict(model, va_loader)
    test_pred += predict(model, te_loader) / N_SPLITS
    fold_scores.append(float(roc_auc_score(y[va_i], oof[va_i])))
    el = time.time() - t_start
    print(f"fold {f}: AUC {fold_scores[-1]:.6f}   elapsed {el/60:.1f} min, "
          f"about {el/(f+1)*(N_SPLITS-f-1)/60:.1f} min left\n")

cv_mean, cv_std = float(np.mean(fold_scores)), float(np.std(fold_scores))
print(f"neural CV {cv_mean:.6f} +/- {cv_std:.6f} in {(time.time()-t_start)/60:.1f} min")
print(f"  LightGBM 5-seed blend for reference: {LGB_BEST_CV:.6f}")
print(f"  difference: {cv_mean - LGB_BEST_CV:+.6f}")
print("\nA negative difference here is expected and is NOT the decision. The blend is.")

## Save, and blend if the LightGBM out-of-fold vector is attached

The blend gate needs `lgbm_bag08_seedblend5_oof.npy` from the local repo. Attach it as a
Kaggle dataset and the gate runs here; otherwise download `neural_oof.npy` from this
notebook's output and run the comparison locally. Either way the artifacts below are the
deliverable.

In [ ]:
np.save(OUT / "neural_oof.npy", oof)
np.save(OUT / "neural_test.npy", test_pred)
sub = sample.copy()
sub[TARGET] = test_pred
sub.to_csv(OUT / "submission.csv", index=False)
print(f"wrote neural_oof.npy, neural_test.npy, submission.csv to {OUT}")
print(f"fold alignment was {'verified' if ALIGNED else 'NOT VERIFIED'}, "
      f"CV {cv_mean:.6f} +/- {cv_std:.6f}")

found = None
for p in Path("/kaggle/input").glob("*/*.npy") if Path("/kaggle/input").exists() else []:
    if "seedblend5" in p.name or "lgbm" in p.name:
        found = p
        break

if found is None:
    print("\nNo LightGBM OOF attached. Download neural_oof.npy and run the gate locally.")
else:
    lgb_oof = np.load(found)
    print(f"\nloaded {found.name}")
    rank = lambda v: pd.Series(v).rank(pct=True).to_numpy()
    blend = 0.5 * rank(lgb_oof) + 0.5 * rank(oof)
    per_fold = lambda v: np.array([roc_auc_score(y[folds == f], v[folds == f])
                                   for f in range(N_SPLITS)])
    b, l = per_fold(blend), per_fold(lgb_oof)
    d = b - l
    print(f"LightGBM alone : {l.mean():.6f}")
    print(f"50/50 blend    : {b.mean():.6f}")
    print(f"gain           : {d.mean():+.6f}, paired sd {d.std(ddof=1):.6f}, "
          f"wins {int((d > 0).sum())}/{N_SPLITS} folds")
    print(f"Spearman vs LightGBM: "
          f"{float(np.corrcoef(rank(lgb_oof), rank(oof))[0, 1]):.4f} "
          f"(descriptive only, it does not predict blend value on this data)")
    if d.mean() > 0 and int((d > 0).sum()) == N_SPLITS:
        print("\nVERDICT: proceed. Blend it, tune the weight, submit.")
    elif d.mean() > 0:
        print("\nVERDICT: marginal. Positive on average but not on every fold.")
    else:
        print("\nVERDICT: stop. The neural family does not help either, and that closes")
        print("the last diversity hypothesis. Say so in NOTES.md and spend the")
        print("remaining time on the writeup rather than on more models.")

## What this changed

Filled in after the run, and promoted into `NOTES.md` whichever way it goes. A neural
model that fails to blend is a real result: it closes the last hypothesis on the board
and redirects the remaining time to the four flywheel artifacts, which are worth more
than another 0.001 either way.